In [2]:
import numpy as np
import pandas as pd

# 1. Load Raw API Extraction
raw_df = pd.read_csv(r"F:\ASC 23-27\Sem 7\Business Analytics\Case Study\Final cs\AgriProcure\AgriProcure\data\preprocessed dataset.csv")
print(f"Loaded raw dataset with shape: {raw_df.shape}")

# 2. Data Type Conversions & Temporal Parsing
raw_df["date"] = pd.to_datetime(raw_df["date"])
raw_df["crop"] = raw_df["crop"].astype("category")
raw_df["state"] = raw_df["state"].astype("category")
raw_df["market_name"] = raw_df["market_name"].astype("category")

# 3. Drop Redundant ETL Metadata
drop_cols = ["fetched_at", "in_shortlist", "source"]
clean_daily_df = raw_df.drop(
    columns=[c for c in drop_cols if c in raw_df.columns]
).copy()

# 4. Domain Logic & Data Integrity Assertions
assert (
    clean_daily_df["min_price_rs_quintal"]
    <= clean_daily_df["modal_price_rs_quintal"]
).all(), "Error: min_price > modal_price detected!"
assert (
    clean_daily_df["modal_price_rs_quintal"]
    <= clean_daily_df["max_price_rs_quintal"]
).all(), "Error: modal_price > max_price detected!"
assert (clean_daily_df["arrivals_tonnes"] > 0).all(), (
    "Error: zero or negative arrivals detected!"
)

# 5. Outlier Audit via IQR (Price Spread & Arrivals)
clean_daily_df["price_spread"] = (
    clean_daily_df["max_price_rs_quintal"]
    - clean_daily_df["min_price_rs_quintal"]
)


def flag_outliers_iqr(df, col, group_col="crop"):
    """Identifies statistical anomalies by crop category."""
    q1 = df.groupby(group_col)[col].transform(lambda x: x.quantile(0.25))
    q3 = df.groupby(group_col)[col].transform(lambda x: x.quantile(0.75))
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return (df[col] < lower_bound) | (df[col] > upper_bound)


clean_daily_df["is_price_spread_outlier"] = flag_outliers_iqr(
    clean_daily_df, "price_spread"
)
clean_daily_df["is_arrival_outlier"] = flag_outliers_iqr(
    clean_daily_df, "arrivals_tonnes"
)

outlier_arrivals_count = clean_daily_df["is_arrival_outlier"].sum()
print(
    f"Identified {outlier_arrivals_count} volume shock records (retained as genuine seasonal supply events)."
)

# 6. Core Scope Filtering (Focus on primary volatile crops: Tomato, Onion, Potato)
target_crops = ["Tomato", "Onion", "Potato"]
top_crops_df = clean_daily_df[
    clean_daily_df["crop"].isin(target_crops)
].copy()

# 7. Build Market x Crop x Week Panel (Task 7 & Rubric Requirement)
# Assign Monday of the corresponding calendar week
top_crops_df["week_start_date"] = (
    top_crops_df["date"]
    - pd.to_timedelta(top_crops_df["date"].dt.dayofweek, unit="D")
)

weekly_panel = (
    top_crops_df.groupby(
        ["crop", "state", "district", "market_name", "week_start_date"],
        observed=True,
    )
    .agg(
        weekly_modal_price=("modal_price_rs_quintal", "mean"),
        weekly_min_price=("min_price_rs_quintal", "min"),
        weekly_max_price=("max_price_rs_quintal", "max"),
        weekly_total_arrivals=("arrivals_tonnes", "sum"),
        weekly_avg_daily_arrivals=("arrivals_tonnes", "mean"),
        trading_days_active=("date", "count"),
    )
    .reset_index()
)

# Round price metrics for practical reporting
weekly_panel["weekly_modal_price"] = weekly_panel["weekly_modal_price"].round(2)
weekly_panel["weekly_avg_daily_arrivals"] = weekly_panel[
    "weekly_avg_daily_arrivals"
].round(2)
weekly_panel["weekly_total_arrivals"] = weekly_panel[
    "weekly_total_arrivals"
].round(2)

print(
    f"Constructed weekly panel: {weekly_panel.shape[0]} rows across {weekly_panel['market_name'].nunique()} mandis."
)

# 8. Export Processed Artifacts
clean_daily_df.to_parquet("cleaned_daily_mandi.parquet", index=False)
weekly_panel.to_csv("mandi_crop_week_panel.csv", index=False)
print("Saved artifacts: 'cleaned_daily_mandi.parquet' & 'mandi_crop_week_panel.csv'")

Loaded raw dataset with shape: (241780, 12)


C:\Users\aathi\AppData\Local\Temp\ipykernel_28652\3437065533.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q1 = df.groupby(group_col)[col].transform(lambda x: x.quantile(0.25))
C:\Users\aathi\AppData\Local\Temp\ipykernel_28652\3437065533.py:43: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  q3 = df.groupby(group_col)[col].transform(lambda x: x.quantile(0.75))
C:\Users\aathi\AppData\Local\Temp\ipykernel_28652\3437065533.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tru

Identified 0 volume shock records (retained as genuine seasonal supply events).
Constructed weekly panel: 11130 rows across 70 mandis.
Saved artifacts: 'cleaned_daily_mandi.parquet' & 'mandi_crop_week_panel.csv'
